In [ ]:
from huggingface_hub import login
from transformers import AutoModel 
import torch
from conch.open_clip_custom import create_model_from_pretrained, tokenize, get_tokenizer
from PIL import Image

login()

/home/ubuntu/miniconda3/envs/titan/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from conch.open_clip_custom import create_model_from_pretrained
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model, preprocess = create_model_from_pretrained('conch_ViT-B-16', "hf_hub:MahmoodLab/conch",device=device)
_ = model.eval()

In [ ]:
import os
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

cell_abbreviation_mapping = {
    "ABE": "Abnormal eosinophil",
    "ART": "Artefact",
    "BAS": "Basophil",
    "BLA": "Blast",
    "EBO": "Erythroblast",
    "EOS": "Eosinophil",
    "FGC": "Faggott cell",
    "HAC": "Hairy cell",
    "KSC": "Smudge cell",
    "LYI": "Immature lymphocyte",
    "LYT": "Lymphocyte",
    "MMZ": "Metamyelocyte",
    "MON": "Monocyte",
    "MYB": "Myelocyte",
    "NGB": "Band neutrophil",
    "NGS": "Segmented neutrophil",
    "NIF": "Not identifiable",
    "OTH": "Other cell",
    "PEB": "Proerythroblast",
    "PLM": "Plasma cell",
    "PMO": "Promyelocyte"
}


root = '/lustre/groups/labs/marr/qscd01/projects/cytology_vlm_eval/Datasets'
for data in ['Acevedo', 'HiCervix', 'Bone_Marrow_Cyto']:
    data_root = os.path.join(root, data, 'test')
    label_csv = pd.read_csv(os.path.join(data_root, f'{data}_test_labels.csv'))
    labels = label_csv.label.unique()
    if data == 'Bone_Marrow_Cyto':
        labels = [cell_abbreviation_mapping[a] for a in labels]
    prompts = [f'an image of {l}' for l in labels]

    tokenizer = get_tokenizer()
    tokenized_prompts = tokenize(texts=prompts, tokenizer=tokenizer).to(device)

    names = []
    predictions = []
    
    for image_name in tqdm(label_csv.image_name):
        image_name = f'{image_name}.jpg'
        image = Image.open(os.path.join(data_root, image_name))
        image_tensor = preprocess(image).unsqueeze(0).to(device)

        with torch.inference_mode():
            image_embeddings = model.encode_image(image_tensor)
            text_embeddings = model.encode_text(tokenized_prompts)
            sim_scores = (image_embeddings @ text_embeddings.T * model.logit_scale.exp()).softmax(dim=-1).cpu().numpy()
            prediction = labels[sim_scores.argmax()]
        
        names.append(image_name)
        predictions.append(prediction)

    df = pd.DataFrame({'image_name': names, 'cell_type': predictions})
    output_path = os.path.join(f'{data}_0shot_classification_conch_answers.csv')
    df.to_csv(output_path, index=False)


100%|██████████| 979/979 [00:36<00:00, 26.58it/s]


In [ ]:
data_root = '/lustre/groups/labs/marr/qscd01/projects/cytology_vlm_eval/Datasets/WBCAtt/test'
label_csv = pd.read_csv(os.path.join(data_root, 'WBCAtt_test_labels.csv'))
tasks = label_csv.columns[2:]

tasks_WBCAtt_0shot_classification = {
    'label': ['Neutrophil', 'Eosinophil', 'Basophil', 'Lymphocyte', 'Monocyte'],
    'cell_size': ['big', 'small'],
    'cell_shape': ['round', 'irregular'],
    'nucleus_shape': [
        'unsegmented-band', 'unsegmented-round', 'segmented-multilobed', 
        'segmented-bilobed', 'irregular', 'unsegmented-indented'
    ],
    'nuclear_cytoplasmic_ratio': ['low', 'high'],
    'chromatin_density': ['densely', 'loosely'],
    'cytoplasm_vacuole': ['no', 'yes'],
    'cytoplasm_texture': ['clear', 'frosted'],
    'cytoplasm_colour': ['light blue', 'blue', 'purple blue'],
    'granule_type': ['small', 'round', 'coarse', 'nil'],
    'granule_colour': ['pink', 'purple', 'red', 'nil'],
    'granularity': ['yes', 'no']
}
prompts_WBCAtt_0shot_classification = {
    'label': 'an image of ',
    'cell_size': 'the cell size is ',
    'cell_shape': 'the cell shape is ',
    'nucleus_shape': 'the nucleus shape is ',
    'nuclear_cytoplasmic_ratio': 'the nuclear cytoplasmic ratio is ',
    'chromatin_density': 'the chromatin density is ',
    'cytoplasm_vacuole': 'the cytoplasm vacuole: ',
    'cytoplasm_texture': 'the cytoplasm texture is ',
    'cytoplasm_colour': 'the cytoplasm color is ',
    'granule_type': 'the granule type is ',
    'granule_colour': 'the granule color is ',
    'granularity': 'the granularity: '
}
dfs =[]
for task in tasks:
    labels = tasks_WBCAtt_0shot_classification[task]
    prompt = prompts_WBCAtt_0shot_classification[task]
    prompts = [prompt+l for l in labels]
    
    tokenizer = get_tokenizer()
    tokenized_prompts = tokenize(texts=prompts, tokenizer=tokenizer).to(device)

    names = []
    predictions = []
    
    for image_name in tqdm(label_csv['image_name']):
        image_name = f'{image_name}.jpg'
        image = Image.open(os.path.join(data_root, image_name))
        image_tensor = preprocess(image).unsqueeze(0).to(device)

        with torch.inference_mode():
            image_embeddings = model.encode_image(image_tensor)
            text_embeddings = model.encode_text(tokenized_prompts)
            sim_scores = (image_embeddings @ text_embeddings.T * model.logit_scale.exp()).softmax(dim=-1).cpu().numpy()
            prediction = labels[sim_scores.argmax()]
        
        names.append(image_name)
        predictions.append(prediction)

    dfs.append(pd.DataFrame({'image_name': names, task: predictions}))

df = pd.DataFrame({'image_name': names})
for d in dfs:
    df = df.merge(d,on='image_name',how='inner')
output_path = os.path.join('WBCAtt_0shot_classification_conch_answers.csv')
df.to_csv(output_path, index=False)
    


  0%|          | 0/250 [00:00<?, ?it/s]

100%|██████████| 250/250 [00:03<00:00, 62.54it/s]
